# Conv3四路并行 RTL CNN：1.1.0候选
完整整机实现及离线数值验证已通过；新实板功能与性能待验证。独立目录、新内核、真实断电后按格启动，不要Run All。


In [ ]:
from pathlib import Path
import sys
PROJECT_ROOT=Path.cwd().resolve()
if PROJECT_ROOT.name=='notebooks': PROJECT_ROOT=PROJECT_ROOT.parent
if not (PROJECT_ROOT/'project.json').is_file(): raise RuntimeError('从本工程notebooks目录打开本Notebook')
if 'psv7_pynq' in sys.modules and Path(sys.modules['psv7_pynq'].__file__).resolve().parent!=PROJECT_ROOT:
    raise RuntimeError('内核加载了另一工程；先正常停止旧应用，再使用本Notebook的新内核')
if 'session' in globals() and any(getattr(session,k,None) is not None for k in ('app','audio','host_service','_process_lock')):
    raise RuntimeError('已有会话尚未完整停止，请保留当前内核和Owner')
sys.path.insert(0,str(PROJECT_ROOT.parent))
from psv7_pynq.app.session import PynqSession
session=PynqSession(PROJECT_ROOT)
session.check()


## 启动一次
当前已知旧应用正常停止且未断电时保留reuse/False。真正断电上电才改cold_start/True。运行后应自动显示PSV7_CAMERA_HANDOFF_READY且requested/observed均为OVERLAP_SECOND_EVENT_WITH_PARK；刷新返回的网页地址并连接。无需执行app.set_camera_handoff_policy。


In [ ]:
STARTUP_MODE='reuse'  # 旧应用正常停止且未断电：reuse；真正断电上电后：cold_start
CLEAN_BOARD_POWER_UP=False  # 真正断电重新上电才用True，并配合cold_start
BOARD_IP='192.168.2.99'  # 改用其它网口IP时同步修改
app=session.start(mode=STARTUP_MODE,clean_boot=CLEAN_BOARD_POWER_UP)
try:
    host_info=session.start_host(host='0.0.0.0',port=8765,allowed_hosts=[BOARD_IP])
except Exception:
    session.stop()
    raise
{'version':session.config['project_version'],
 'camera_handoff_policy':app.camera_handoff_policy,
 'host':host_info}


## 网页真实摄像一次
选一张实际已知纸片，网页切摄像并识别，等结果结束。下面EXPECTED_PAPER_NAME按真实纸片填写（默认乙酸），不是按预测改名。观察网页/HDMI名称、模型及本次摄像计时对应，然后运行下格留证。


In [ ]:
from psv7_pynq.tools.entry_validation import collect_entry_state
EXPECTED_PAPER_NAME='乙酸'
camera_entry=collect_entry_state(session,'CAMERA',EXPECTED_PAPER_NAME)
camera_entry


## 网页实麦一次
网页切语音，等SPEAK_NOW后说一个实际已知名称（默认乙酸），等结果完成。下面按实际说词填写；观察两端名称/模型及语音计算计时对应，再运行下格。


In [ ]:
EXPECTED_SPOKEN_NAME='乙酸'
voice_entry=collect_entry_state(session,'VOICE',EXPECTED_SPOKEN_NAME)
voice_entry


## 清除并切回摄像
网页清除结果，再切回摄像，确认画面恢复、旧名称/模型/两种计时消失。运行下格留证。这里验证模式转换后自动策略仍保持，不重做已接受的采音active补充。


In [ ]:
empty_entry=collect_entry_state(session,'CAMERA',expect_empty=True)
empty_entry


## 正常停止
运行下格，回传本轮三项ENTRY_DELTA_SNAPSHOT及完整停止输出，并确认两端实际观察。候选检查通过之前不要删除原0.7.3目录。


In [ ]:
stop_report=session.stop()
stop_report
